# 5.4 · 朴素贝叶斯 / Naive Bayes

> **课程定位 / Where this fits**
> 前几课直接建模决策边界(判别式)。朴素贝叶斯是**生成式**模型代表: 用贝叶斯定理(2.8)对 $\Pr(类\mid特征)$ 建模, "朴素"地假设**特征条件独立**。简单、快、在文本分类(垃圾邮件)上出奇地好。
> Naive Bayes is the canonical generative classifier: Bayes' theorem (2.8) plus the "naive" conditional-independence assumption. Simple, fast, surprisingly strong on text.

> 💡 **面试相关 / Interview-relevant**
> - "朴素贝叶斯的'朴素'假设是什么 / 为什么仍然有效" ★★★★★
> - "Gaussian / Multinomial / Bernoulli NB 区别" ★★★★
> - "拉普拉斯平滑解决什么问题" ★★★★（零概率）
> - "为什么在对数空间计算" ★★★★（下溢）
> - "生成式 vs 判别式" ★★★★

---

## 学习目标 / Learning Objectives
1. 用贝叶斯定理推出朴素贝叶斯, 看清"朴素"假设。
2. 三种变体: Gaussian(连续)/ Multinomial(计数)/ Bernoulli(0-1)。
3. **拉普拉斯平滑**防零概率 + **对数空间**防下溢。
4. 从零实现 + 文本垃圾短信分类。
5. 生成式 vs 判别式。

## 目录 / TOC
1. [贝叶斯定理到 NB ⭐](#1)
2. [三种变体 ⭐](#2)
3. [📧 数据: SMS 垃圾短信](#3)
4. [从零 Multinomial NB + 平滑 + 对数 ⭐](#4)
5. [对照 sklearn + 高斯NB on Iris](#5)
6. [生成式 vs 判别式 ⭐](#6)
7. [小结](#7)


<a id="1"></a>
## 1. 贝叶斯定理到朴素贝叶斯 ⭐ / Bayes → Naive Bayes

贝叶斯定理(2.8): 给定特征 $\mathbf{x}=(x_1,\dots,x_d)$, 类 $c$ 的后验
$$\Pr(c\mid\mathbf{x}) = \frac{\Pr(c)\,\Pr(\mathbf{x}\mid c)}{\Pr(\mathbf{x})} \propto \Pr(c)\,\Pr(\mathbf{x}\mid c)$$

难点是似然 $\Pr(\mathbf{x}\mid c)=\Pr(x_1,\dots,x_d\mid c)$ —— 联合分布维度爆炸。

**"朴素"假设**: 给定类别, 各特征**条件独立**：
$$\Pr(\mathbf{x}\mid c) = \prod_{j=1}^{d} \Pr(x_j\mid c)$$

于是分类规则(取后验最大, MAP):
$$\hat{c} = \arg\max_c \;\Pr(c)\prod_j \Pr(x_j\mid c)$$

**为什么"明显错误"的独立假设仍有效**(面试爱问): 即使概率估得不准, 只要**正确类的后验仍最大**, 分类就对。NB 要的是排序对, 不是概率准。


<a id="2"></a>
## 2. 三种变体 ⭐ / Three Variants

区别只在**如何建模 $\Pr(x_j\mid c)$**：

| 变体 | 特征类型 | $\Pr(x_j\mid c)$ 模型 | 典型场景 |
|---|---|---|---|
| **Gaussian** | 连续 | 正态 $\mathcal{N}(\mu_{jc},\sigma_{jc}^2)$ | Iris 等数值特征 |
| **Multinomial** | 计数 | 多项(词频) | 文本(词袋计数) |
| **Bernoulli** | 0/1 | 伯努利(出现与否) | 文本(词是否出现) |


<a id="3"></a>
## 3. 数据: SMS 垃圾短信 / SMS Spam (mini)

内联一个小型双语垃圾短信集(同 3.7 的风格)。任务: 判断短信是 spam(垃圾) 还是 ham(正常)。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

spam = [
    "WIN a FREE prize now click here", "free entry to win cash prize", "claim your free reward urgent",
    "you won a lottery call now", "cheap loans apply free today", "free gift card click link now",
    "urgent win money prize claim", "congratulations you won free cash", "limited offer free click now",
    "win free iphone click here urgent",
]
ham = [
    "are we still meeting for lunch today", "can you send me the report please", "happy birthday see you tonight",
    "the meeting is moved to 3pm", "thanks for your help yesterday", "let me know when you arrive home",
    "i will call you after work today", "please review the document when free", "see you at the gym later",
    "did you finish the homework yet",
]
texts = spam + ham
labels = np.array([1]*len(spam) + [0]*len(ham))   # 1=spam
print(f"SMS: {len(texts)} 条 ({labels.sum()} spam / {(labels==0).sum()} ham)")
print("spam 例:", spam[0]); print("ham  例:", ham[0])


<a id="4"></a>
## 4. 从零 Multinomial NB + 平滑 + 对数 ⭐ / From Scratch

**两个工程关键**：
- **拉普拉斯平滑**: 若某词在训练中从未在某类出现, $\Pr(词\mid c)=0$, 整个连乘归零。加 $\alpha$(常=1)平滑: $\frac{\text{count}+\alpha}{\text{total}+\alpha V}$, 给未见词一点概率。
- **对数空间**: 很多小概率连乘会**下溢**到 0。取对数把连乘变连加: $\log\Pr(c)+\sum_j\log\Pr(x_j\mid c)$。


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split

vec = CountVectorizer()
Xc = vec.fit_transform(texts).toarray()   # 词频矩阵
Xtr, Xte, ytr, yte = train_test_split(Xc, labels, test_size=0.3, stratify=labels, random_state=0)

class MultinomialNB_scratch:
    def __init__(self, alpha=1.0): self.alpha = alpha
    def fit(self, X, y):
        self.classes = np.unique(y)
        self.log_prior = {}; self.log_lik = {}
        V = X.shape[1]
        for c in self.classes:
            Xc = X[y == c]
            self.log_prior[c] = np.log(len(Xc) / len(X))
            counts = Xc.sum(axis=0) + self.alpha            # 拉普拉斯平滑
            self.log_lik[c] = np.log(counts / counts.sum()) # 对数空间
        return self
    def predict(self, X):
        out = []
        for x in X:
            scores = {c: self.log_prior[c] + (x * self.log_lik[c]).sum()  # 连加=对数连乘
                      for c in self.classes}
            out.append(max(scores, key=scores.get))
        return np.array(out)

nb = MultinomialNB_scratch().fit(Xtr, ytr)
print(f"从零 Multinomial NB 准确率: {(nb.predict(Xte) == yte).mean():.3f}")

# 看最"spam-y"的词 / most spammy words
log_ratio = nb.log_lik[1] - nb.log_lik[0]
top = np.argsort(log_ratio)[-6:][::-1]
words = np.array(vec.get_feature_names_out())
print("最像 spam 的词:", list(words[top]))


<a id="5"></a>
## 5. 对照 sklearn + 高斯 NB on Iris / sklearn & Gaussian NB


In [ ]:
from sklearn.naive_bayes import MultinomialNB, GaussianNB
sk = MultinomialNB(alpha=1.0).fit(Xtr, ytr)
print(f"sklearn MultinomialNB 准确率: {sk.score(Xte, yte):.3f}  (与从零一致)")

# Gaussian NB 处理连续特征 (Iris) / Gaussian NB on continuous Iris
from sklearn.datasets import load_iris
iris = load_iris()
ix_tr, ix_te, iy_tr, iy_te = train_test_split(iris.data, iris.target, test_size=0.3,
                                              stratify=iris.target, random_state=0)
gnb = GaussianNB().fit(ix_tr, iy_tr)
print(f"Gaussian NB on Iris 准确率: {gnb.score(ix_te, iy_te):.3f}")
print("Gaussian NB 假设每类每特征服从正态, 用训练集的 μ,σ 估计")
print("注: NB 不需要缩放(每特征独立建模), 这点和 KNN/SVM 不同")


<a id="6"></a>
## 6. 生成式 vs 判别式 ⭐ / Generative vs Discriminative

| | 生成式 (NB, LDA) | 判别式 (逻辑回归, SVM) |
|---|---|---|
| 建模对象 | 联合 $\Pr(\mathbf{x}, c)$ → 反推后验 | 直接 $\Pr(c\mid\mathbf{x})$ 或边界 |
| 数据少时 | 收敛快、更稳 | 易过拟合 |
| 数据多时 | 受错误假设拖累 | 通常更准 |
| 副产品 | 能**生成**新样本 | 不能 |

**经典结论**(Ng & Jordan): 小数据 NB 往往更好, 大数据逻辑回归追上并超过。NB 训练**极快**(只数频次), 是高维文本的好基线。


In [ ]:
from sklearn.linear_model import LogisticRegression
print("文本任务上 NB vs 逻辑回归(本小数据):")
print(f"  MultinomialNB:      {sk.score(Xte, yte):.3f}")
print(f"  LogisticRegression: {LogisticRegression(max_iter=1000).fit(Xtr,ytr).score(Xte,yte):.3f}")
print("生成式(NB)只数频次, 训练 O(数据量); 判别式(LR)要迭代优化")


<a id="7"></a>
## 7. 小结 / Summary

```
朴素贝叶斯: P(c|x) ∝ P(c)∏P(xⱼ|c) (贝叶斯 2.8 + 条件独立"朴素"假设)
即使独立假设错, 只要正确类后验最大 → 分类仍对
变体: Gaussian(连续) / Multinomial(计数,文本) / Bernoulli(0-1)
工程: 拉普拉斯平滑防零概率; 对数空间防下溢
不需缩放; 训练极快(数频次); 高维文本强基线
生成式(建模联合, 能生成) vs 判别式(直接建边界): 小数据 NB 优, 大数据 LR 优
```

### 💡 面试速查
1. **"朴素"=给定类别特征条件独立**; 假设常错但分类仍有效(只需排序对)
2. **三变体**按特征类型选: 连续→Gaussian, 词频→Multinomial, 0/1→Bernoulli
3. **拉普拉斯平滑**防未见词把连乘归零; **对数空间**防下溢
4. **生成式 vs 判别式**: NB 训练快、小数据稳, LR 大数据更准

### 下一节
**5.5 SVM**——回到判别式。SVM 用"最大间隔"找最稳健的边界, 核技巧让它能画非线性边界, 曾是深度学习前的王者。
